<a href="https://colab.research.google.com/github/sheriannmclarty/DATA620_WebAnalytics_SheriannMc/blob/main/projects/name_gender_corpus_skm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Name Gender Classification Using the NLTK Names Corpus
**DATA 620 | Project 3 | Spring 2026**

**Author:** Sheriann McLarty

https://github.com/sheriannmclarty/DATA620_WebAnalytics_SheriannMc

This notebook builds a name gender classifier using the NLTK Names Corpus.
Starting from a single-feature baseline, we incrementally improved performance
through feature engineering and compared three scikit-learn classifiers.
The final model was evaluated on a held-out test set.




## Load Libraries

In [16]:
import random
import nltk
from nltk.corpus import names

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

nltk.download('names')

[nltk_data] Downloading package names to /root/nltk_data...
[nltk_data]   Package names is already up-to-date!


True

## 1. Load and Split the Names Corpus

The NLTK Names Corpus contains lists of male and female first names. I labeled each name as either `"male"` or `"female"`, shuffled the dataset using a fixed random seed for reproducibility, and split the data into training, dev-test, and test sets.

The original assignment prompt mentions 6,900 names for training, but the current NLTK corpus actually contains more than that overall, so after setting aside 500 names for test and 500 for dev-test, the remaining names form the training set.

In [17]:
labeled_names = ([(name, 'male') for name in names.words('male.txt')] +
                 [(name, 'female') for name in names.words('female.txt')])

random.seed(42)
random.shuffle(labeled_names)

test_data = labeled_names[:500]
devtest_data = labeled_names[500:1000]
train_data = labeled_names[1000:]

print("Total names:", len(labeled_names))
print("Training set:", len(train_data))
print("Dev-test set:", len(devtest_data))
print("Test set:", len(test_data))

Total names: 7944
Training set: 6944
Dev-test set: 500
Test set: 500


## 2. Baseline Feature Set

I started with the classic baseline feature from the NLTK example: the **last letter** of the name. This is a reasonable starting point because many names show gendered patterns in their endings.

I used this baseline first so I could compare it against stronger feature sets later and see whether my additions actually improved performance.

In [18]:
def baseline_features(name):
    name = name.lower()
    return {
        "last_letter": name[-1]
    }

In [19]:
def prepare_features(data, feature_func):
    X = [feature_func(name) for name, label in data]
    y = [label for name, label in data]
    return X, y

X_train_base_dict, y_train = prepare_features(train_data, baseline_features)
X_dev_base_dict, y_dev = prepare_features(devtest_data, baseline_features)
X_test_base_dict, y_test = prepare_features(test_data, baseline_features)

vec_base = DictVectorizer(sparse=True)
X_train_base = vec_base.fit_transform(X_train_base_dict)
X_dev_base = vec_base.transform(X_dev_base_dict)
X_test_base = vec_base.transform(X_test_base_dict)

baseline_model = LogisticRegression(max_iter=2000)
baseline_model.fit(X_train_base, y_train)

baseline_dev_pred = baseline_model.predict(X_dev_base)
baseline_test_pred = baseline_model.predict(X_test_base)

print("Baseline dev-test accuracy:", round(accuracy_score(y_dev, baseline_dev_pred), 4))
print("Baseline test accuracy:", round(accuracy_score(y_test, baseline_test_pred), 4))

Baseline dev-test accuracy: 0.756
Baseline test accuracy: 0.746


## 3. Improved Feature Engineering

After establishing a baseline, I added a richer feature set. I wanted to make this part a little more original, so I included features based on the structure of names, not just the ending.

The improved features include:
- first letter
- last letter
- last two letters
- last three letters
- name length
- vowel count
- whether the name starts with a vowel
- whether the name ends with a vowel
- whether the name contains a double letter
- how many double-letter sequences appear in the name

I was especially interested in whether **double letters** or other spelling patterns might help capture differences that the baseline model would miss.

In [20]:
def custom_features(name):
    name = name.lower()
    vowels = "aeiou"

    return {
        "first_letter": name[0],
        "last_letter": name[-1],
        "last_two": name[-2:] if len(name) >= 2 else name,
        "last_three": name[-3:] if len(name) >= 3 else name,
        "name_length": len(name),
        "vowel_count": sum(1 for c in name if c in vowels),
        "starts_with_vowel": name[0] in vowels,
        "ends_with_vowel": name[-1] in vowels,
        "has_double_letter": any(name[i] == name[i+1] for i in range(len(name)-1)),
        "double_letter_count": sum(1 for i in range(len(name)-1) if name[i] == name[i+1]),
    }

In [21]:
X_train_dict, y_train = prepare_features(train_data, custom_features)
X_dev_dict, y_dev = prepare_features(devtest_data, custom_features)
X_test_dict, y_test = prepare_features(test_data, custom_features)

vec = DictVectorizer(sparse=True)
X_train = vec.fit_transform(X_train_dict)
X_dev = vec.transform(X_dev_dict)
X_test = vec.transform(X_test_dict)

print("Number of engineered features:", X_train.shape[1])

Number of engineered features: 1668


## 4. Compare Multiple Classifiers

Since my professor said we were free to use other classification models, I compared several **scikit-learn** classifiers instead of staying only with the NLTK-built ones.

The models I compared were:
- **Logistic Regression**
- **Linear SVM**
- **Random Forest**

I used the **dev-test set** to compare performance and decide which model to carry forward to the final test evaluation.

In [22]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Linear SVM": LinearSVC(),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42)
}

dev_results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    dev_pred = model.predict(X_dev)
    dev_acc = accuracy_score(y_dev, dev_pred)
    dev_results.append((model_name, dev_acc))

dev_results_df = pd.DataFrame(dev_results, columns=["Model", "Dev-Test Accuracy"]).sort_values(
    by="Dev-Test Accuracy", ascending=False
)

dev_results_df

,Model,Dev-Test Accuracy
0,Logistic Regression,0.810
1,Linear SVM,0.790
2,Random Forest,0.788


## 5. Select the Best Model

Based on dev-test performance, I selected the best-performing model for final evaluation on the test set. This is an important step because the dev-test set is meant for model comparison and feature tuning, while the test set should be saved for the final check on unseen data.

In [23]:
best_model_name = dev_results_df.iloc[0]["Model"]
print("Best model on dev-test:", best_model_name)

best_model = models[best_model_name]
best_model.fit(X_train, y_train)

dev_pred_final = best_model.predict(X_dev)
test_pred_final = best_model.predict(X_test)

dev_accuracy = accuracy_score(y_dev, dev_pred_final)
test_accuracy = accuracy_score(y_test, test_pred_final)

print("Final dev-test accuracy:", round(dev_accuracy, 4))
print("Final test accuracy:", round(test_accuracy, 4))

Best model on dev-test: Logistic Regression
Final dev-test accuracy: 0.81
Final test accuracy: 0.804


## 6. Final Classification Report

To better understand the final model’s performance, I also looked at the classification report for the test set. This gives a more detailed view of precision, recall, and F1-score for the male and female classes.

In [24]:
print(classification_report(y_test, test_pred_final))

              precision    recall  f1-score   support

      female       0.83      0.87      0.85       316
        male       0.76      0.69      0.72       184

    accuracy                           0.80       500
   macro avg       0.79      0.78      0.79       500
weighted avg       0.80      0.80      0.80       500



In [25]:
cm = confusion_matrix(y_test, test_pred_final, labels=["male", "female"])
cm_df = pd.DataFrame(cm, index=["Actual Male", "Actual Female"], columns=["Predicted Male", "Predicted Female"])
cm_df

,Predicted Male,Predicted Female
Actual Male,127,57
Actual Female,41,275


## 7. Discussion

The improved model performed better than the baseline, increasing accuracy from 75.6% to about 81%. This suggests that the added structural features captured useful information beyond the final letter alone. Features such as the last two or three letters, vowel count, and double-letter patterns helped the classifier learn more of the spelling structure of names rather than relying only on a single ending character.

One of the more interesting findings from the error analysis is that many of the misclassified names end in **"e"** or **"y"**, including names such as *Sayre, Kingsley, Garry, Thayne, Lyle, Fonzie, Morlee, Arne, Micky,* and *Roxy*. This suggests that these endings are less reliable indicators of gender than more strongly patterned endings such as **-a**. In other words, the model performs least reliably on names whose spelling patterns are shared across both classes.

The final test accuracy was slightly lower than the dev-test accuracy, which is what I expected. Because I used the dev-test set repeatedly during feature engineering and model comparison, performance on that set is somewhat optimistic. The held-out test set provides a cleaner estimate of generalization to unseen names.

Overall, the project shows that even a simple classification task can benefit from thoughtful feature engineering and model comparison. It also shows that some names remain difficult to classify because their spelling patterns are genuinely less distinct.

## 8. Error Analysis

To better understand where the classifier was less reliable, I reviewed a sample of misclassified names from the test set. A noticeable pattern was that many of these names ended in **"e"** or **"y"**, such as *Sayre, Kingsley, Garry, Thayne, Lyle, Fonzie, Morlee, Arne, Micky,* and *Roxy*. These endings appear in both male and female names, which makes them weaker cues for classification than endings with stronger gender associations.

This suggests that the model performs best when a name contains a more distinctive spelling pattern and is less consistent when the name has endings that are shared across both classes. In that sense, the errors are not random — they reflect genuine ambiguity in the data.



In [26]:
errors = []

for (name, actual_label), predicted_label in zip(test_data, test_pred_final):
    if actual_label != predicted_label:
        errors.append((name, actual_label, predicted_label))

errors[:25]

[('Sayre', 'male', np.str_('female')),
 ('Angel', 'female', np.str_('male')),
 ('Kingsley', 'male', np.str_('female')),
 ('Trixy', 'female', np.str_('male')),
 ('Garry', 'male', np.str_('female')),
 ('Christin', 'female', np.str_('male')),
 ('Siobhan', 'female', np.str_('male')),
 ('Evy', 'female', np.str_('male')),
 ('Pet', 'female', np.str_('male')),
 ('Micky', 'male', np.str_('female')),
 ('Shelagh', 'female', np.str_('male')),
 ('Fonzie', 'male', np.str_('female')),
 ('Sharron', 'female', np.str_('male')),
 ('Kenneth', 'male', np.str_('female')),
 ('Thayne', 'male', np.str_('female')),
 ('Lyle', 'male', np.str_('female')),
 ('Erin', 'male', np.str_('female')),
 ('Pate', 'male', np.str_('female')),
 ('Giorgi', 'male', np.str_('female')),
 ('Morlee', 'male', np.str_('female')),
 ('Arne', 'male', np.str_('female')),
 ('Roxy', 'female', np.str_('male')),
 ('Kin', 'male', np.str_('female')),
 ('Karsten', 'male', np.str_('female')),
 ('Anatoly', 'male', np.str_('female'))]

## 8b. Table of Error Frequency by Letter
To quantify the pattern observed in the error sample, I counted how often each
last letter appeared among all misclassified names. The table below ranks the
most common last letters in errors, showing that names ending in **"e"** and
**"y"** account for a disproportionate share of misclassifications.

In [27]:
import collections

error_endings = collections.Counter(name[-1].lower() for name, actual, predicted in errors)
error_df = pd.DataFrame(error_endings.most_common(), columns=["Last Letter", "Error Count"])
error_df["Pct of Errors"] = (error_df["Error Count"] / len(errors) * 100).round(1)
print(f"Total errors: {len(errors)}")
error_df.head(10)

Total errors: 98


,Last Letter,Error Count,Pct of Errors
0,e,27,27.6
1,n,19,19.4
2,y,14,14.3
3,h,8,8.2
4,l,7,7.1
5,i,6,6.1
6,s,6,6.1
7,t,3,3.1
8,r,3,3.1
9,o,2,2.0


##8c. Chi-Square Test

To confirm that last-letter distributions differ meaningfully between male and
female names, I ran a chi-square test of independence on the training data.
A significant result would validate that last-letter patterns carry real
predictive signal — and help explain why certain endings are more reliable
classifiers than others.

In [28]:
from scipy.stats import chi2_contingency

# Compare last-letter distributions between male and female names
male_endings = collections.Counter(name[-1].lower() for name, label in train_data if label == 'male')
female_endings = collections.Counter(name[-1].lower() for name, label in train_data if label == 'female')

all_letters = sorted(set(male_endings) | set(female_endings))
male_counts = [male_endings.get(l, 0) for l in all_letters]
female_counts = [female_endings.get(l, 0) for l in all_letters]

chi2, p, dof, _ = chi2_contingency([male_counts, female_counts])
print(f"Chi-square statistic: {chi2:.2f}")
print(f"p-value: {'< 0.001' if p < 0.001 else f'{p:.2e}'}")
print("Last-letter distribution differs significantly by gender." if p < 0.05 else "No significant difference found.")

Chi-square statistic: 2322.70
p-value: < 0.001
Last-letter distribution differs significantly by gender.


## 9. Conclusion

For this project, I built a name gender classifier using the NLTK Names Corpus and compared multiple scikit-learn models. Starting from a baseline based only on the last letter, I improved performance by adding more structural features such as double letters, vowel patterns, and name length.

The final model improved substantially over the baseline and achieved about 81% accuracy on the test set. This improvement supports the idea that name structure contains useful predictive information beyond a single suffix feature. At the same time, the error analysis shows that the model still struggles with names ending in more ambiguous patterns such as **"e"** and **"y"**, which appear across both male and female names.

Overall, this project demonstrated that better feature design led to better classification performance, while also highlighting the limits of predicting gender from names alone.

## Next Steps

- Add prefix features (first two or three letters) to complement the suffix-based features already in use
- Extend chi-square analysis to bigrams (last two letters) to identify more granular gender-predictive patterns.
- Explore whether name origin or language of origin could serve as an additional feature
- Test on a more culturally diverse name corpus to evaluate generalization beyond English-dominant naming patterns